In [2]:
from manim import *
import numpy as np
import requests as req
import openai
import time
from tqdm.auto import tqdm
import os

/home/l0stsky/ExTraStuff/Competitions/LearnAnything/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
import subprocess

def merge_folder(folder_path, output_name="final_animation.mp4"):
    if not os.path.exists(folder_path):
        print(f"Error: Folder '{folder_path}' does not exist.")
        return
    video_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.mp4')]

    if not video_files:
        print(f"No MP4 files found in '{folder_path}'.")
        return

    # 2. SORT BY LAST MODIFIED TIME (Oldest first)
    video_files.sort(key=os.path.getmtime) 

    list_file = "scene_list.txt"
    with open(list_file, "w") as f:
        for full_path in video_files:
            safe_path = os.path.abspath(full_path).replace("\\", "/")
            f.write(f"file '{safe_path}'\n")   
    print(f"Merging {len(video_files)} scenes in chronological order:")
    for v in video_files:
        print(f" - {os.path.basename(v)}")
    command = [
        "ffmpeg",
        "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", list_file,
        "-c", "copy",
        output_name
    ]
    try:
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        print(f"\nSuccess! Merged video saved as: {output_name}")
    except subprocess.CalledProcessError:
        print("\nError: FFmpeg failed to merge the videos.")
    finally:
        if os.path.exists(list_file):
            os.remove(list_file)

if __name__ == "__main__":
    # Update this to your actual folder path
    TARGET_FOLDER = "/home/l0stsky/ExTraStuff/Competitions/LearnAnything/media/videos/1080p60/partial_movie_files/CatNeuralNetwork" 
    
    merge_folder(TARGET_FOLDER, "full_network_tutorial.mp4")

In [ ]:
ML_S=[]

In [ ]:
CLIENT = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="JustSomeRandomKey"
)

In [ ]:
def ContactAI(p,sp):
    idx =0
    Unc_C=0
    for _ in range(5):
        idx=idx%len(ML_S)
        Model=ML_S[idx]
        
        try:
            response = CLIENT.chat.completions.create(
                model=Model,
                messages=[
                    {"role": "system", "content": sp},
                    {"role": "user", "content": p}
                ])         
            # r_body=response.read().decode('utf-8') 
            # rs=json.loads(r_body)
            # if rs.get('error'):
                # print(rs['error'])
                # ix+=1
                # continue
            ct = response.choices[0].message.content
            if ct is None:
                print(f"Model {Model} failed, retrying...")
                idx+=1
                time.sleep(1)
                continue
            
            if "Unclear" in ct:
                print("Model Unable to Extract Data, Trying Again")
                if Unc_C>2:
                    return ct
                Unc_C+=1
                idx+=1
                time.sleep(1)
                continue
            return ct
        
        
        except openai.APIError as e:
            print(f"API Error from {Model}: {e}")
            idx += 1
            time.sleep(1)
            
        except Exception as e:
            print(f'Connecting Failed with Error {e}')
            idx +=1
            time.sleep(1)
    else:
        return None


# Layer 1 : Scripter Model

In [ ]:
SYSTEM_PROMPT_S="""You are the Lead Curriculum Architect and Animation Director for an educational video platform. Your job is to take any topic requested by a user and break it down into a logical, highly engaging animated video script.

You do NOT write code. Your output is sent to a downstream Python (Manim) developer model. You must provide the exact structure, the spoken narration, and a concrete visual vision for what should happen on screen.

OUTPUT CONSTRAINTS
You must output your response in STRICT, VALID JSON format.
Output ONLY raw JSON. Do not include markdown formatting, backticks (```), or code blocks. Absolutely NO conversational text, greetings, or explanations before or after the JSON.

1. The Output Structure
Your response must be a single JSON object matching this exact schema:

{
"project_title": "The overarching topic",
"sections": [
{
"section_number": 1,
"topic": "Main idea of this section",
"subtopic": "Specific focus of this scene",
"estimated_duration_seconds": 15,
"narration_script": "The exact, word-for-word text the Text-to-Speech engine will read. Make it conversational, engaging, and clear.",
"animation_vision": "A descriptive, chronological guide of what should happen visually using Manim-friendly geometric concepts."
}
]
}

2. Strict Director Rules

Section Limits: You must divide the topic into at least 2, and NO MORE THAN 7 sections. Keep the flow logical (Hook/Intro -> Core Concepts -> Examples -> Conclusion).

Pacing & Timing: Keep the narration_script concise but impactful. Assume a TTS speaking rate of ~150 words per minute (about 2.5 words per second). Ensure the estimated_duration_seconds accurately reflects the time it takes to read the script, and ensure the visual actions fit within that time frame. Do not write overly academic jargon unless you immediately explain it.

The Animation Vision (Crucial):

This is a creative brief for the downstream Manim coding model. Do NOT write Python code in this field.

Describe the visual flow chronologically.

Frame your visual ideas using Manim-specific actions and geometries (e.g., 'Fade in a large yellow circle in the center', 'Transform the circle into a square', 'Draw a blue arrow pointing right', 'Write text labeled Gravity').

Do not include meta-instructions to the downstream model (e.g., do not say "you can improvise the shapes"). State exactly what you want to see happen on the screen with authority and precision.

3. Error Prevention

Ensure all double quotes inside your JSON strings are properly escaped (").

Ensure the JSON array sections is properly closed.

Never skip a required field. Every section must have all 6 keys.

IF Unable to Generate or NOT Sure, Reply With Unclear and Nothing else"""

In [ ]:
sc=ContactAI(USER_INPUT,SYSTEM_PROMPT_S)

# Layer 2 : Generating MANIN Code

In [ ]:
sects=[i for i in sc["sections"]]

print(f"SCRIPT Generated!!")
print(f"SCRIPT Title : {sc['project_title']}")
print(f"SCRIPT Expected Duration : {sum([i['estimated_duration_seconds'] for i in sc['sections']])}")
print(f"SCRIPT Num Sects : {len(sects)}")

In [ ]:
for sect in tqdm(sects):
    out=ContactAI()

# Testing

In [ ]:
USER_INPUT="Tell me about Neural Networks. Like Idk How Do they Even LEARN ANYTHING? Arent they Just Math??? Also Wtf are those Graph like things with a name with smth like a network."